# WRs

In [ ]:
import pandas as pd, gspread
from gspread_dataframe import set_with_dataframe
from IPython.display import display

CSV_FILE = "2025_LoL_esports_match_data_from_OraclesElixir.csv"
SERVICE_ACCOUNT_FILE = "worlds-475809-69c16d86976c.json"
SHEET_NAME, TAB_NAME = "python access", "Main"

team_order = ["KT","AL","CFO","G2","GEN","TES","HLE","FLY","TSW","BLG","100T","T1","VKS","KOI","FNC","PSG"]
abbr = {
    "KT Rolster":"KT","Anyone's Legend":"AL","CTBC Flying Oyster":"CFO","G2 Esports":"G2","Gen.G":"GEN",
    "Top Esports":"TES","Hanwha Life Esports":"HLE","FlyQuest":"FLY","Team Secret Whales":"TSW",
    "Bilibili Gaming":"BLG","100 Thieves":"100T","T1":"T1","Vivo Keyd Stars":"VKS",
    "Movistar KOI":"KOI","Fnatic":"FNC","PSG Talon":"PSG"
}

def norm(s):
    if pd.isna(s): return s
    return (str(s).replace("’","'").replace("‘","'")
                 .replace("“",'"').replace("”",'"').strip())

# ---------- UPDATED WORLDS TEAM TABLE WITH SERIES LOGIC ----------

# 1) Load + filter
df = pd.read_csv(CSV_FILE, low_memory=False)

for c in ["league","teamname","gameid","result","matchid"]:
    if c in df.columns:
        df[c] = df[c].apply(norm)

worlds = df[df["league"].str.contains(r"\bWLD", case=False, na=False)].copy()

# Drop non-Worlds teams
worlds = worlds[~worlds["teamname"].isin({"JD Gaming","Weibo Gaming","Invictus Gaming"})]

# Keep relevant columns
keep_cols = ["gameid","teamname","result"]
if "matchid" in worlds.columns:
    keep_cols.append("matchid")

worlds = worlds[keep_cols].drop_duplicates()

worlds["result"] = pd.to_numeric(worlds["result"], errors="coerce")
worlds = worlds[worlds["result"].isin([0,1])]

# 2) Opponent per game
pair_map = worlds.groupby("gameid")["teamname"].unique().to_dict()
worlds["opp_team"] = worlds.apply(
    lambda r: next((t for t in pair_map.get(r["gameid"], []) if t != r["teamname"]), None),
    axis=1
)
worlds = worlds.dropna(subset=["opp_team"])

# 3) Series ID
if "matchid" in worlds.columns:
    worlds["series_id"] = worlds["matchid"]
else:
    # remove trailing "_1", "_2", "_3"
    worlds["series_id"] = worlds["gameid"].str.replace(r"_\d+$", "", regex=True)

# 4) Per-series win fractions
series_results = (
    worlds.groupby(["series_id","teamname","opp_team"])["result"]
          .agg(wins="sum", games="count")
          .reset_index()
)

series_results["win_fraction"]  = series_results["wins"] / series_results["games"]
series_results["loss_fraction"] = 1 - series_results["win_fraction"]

# 5) Aggregate into team-level results (series-weighted)
team_records = (
    series_results.groupby("teamname")["win_fraction"]
        .agg(series_wins="sum", series_count="count")
        .reset_index()
)

team_records["Team"]   = team_records["teamname"].map(lambda x: abbr.get(x, x))
team_records["Wins"]   = team_records["series_wins"].round(2)
team_records["Losses"] = (team_records["series_count"] - team_records["series_wins"]).round(2)

team_records["WR"] = (team_records["series_wins"] / team_records["series_count"]).fillna(0)

# 6) Keep exact order
team_records = team_records[team_records["Team"].isin(team_order)].copy()
team_records["Team"] = pd.Categorical(team_records["Team"], categories=team_order, ordered=True)
team_records = team_records.sort_values("Team").reset_index(drop=True)

# 7) Preview table
preview = team_records.loc[:, ["Team","WR","Wins","Losses"]].copy()
preview["WR"] = preview["WR"].map(lambda x: f"{x:.4f}")

print("📤 This exact table will be exported:")
display(preview)

left_block  = preview.loc[:, ["Team","WR"]].copy()
right_block = preview.loc[:, ["Wins","Losses"]].copy()

# ---------- EXPORT TO SHEET (unchanged) ----------

gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sh = gc.open(SHEET_NAME)
ws = sh.worksheet(TAB_NAME)

L_ROW, L_COL = 6, 2   # B6
R_ROW, R_COL = 6, 7   # G6
n = len(preview)

ws.batch_clear([f"B6:C{6+n}", f"G6:H{6+n}"])

set_with_dataframe(ws, left_block,  row=L_ROW, col=L_COL, include_index=False, include_column_header=True)
set_with_dataframe(ws, right_block, row=R_ROW, col=R_COL, include_index=False, include_column_header=True)

print(f"✅ Exported {n} rows to B6:C and G6:H — identical to the preview.")



📤 This exact table will be exported:


,Team,WR,Wins,Losses
0,KT,0.8583,5.15,0.85
1,AL,0.7667,3.07,0.93
2,CFO,0.5556,3.33,2.67
3,G2,0.7167,2.87,1.13
4,GEN,0.6667,4.00,2.00
5,TES,0.4111,2.47,3.53
6,HLE,0.6500,3.25,1.75
7,FLY,0.4667,2.33,2.67
8,TSW,0.3333,1.33,2.67
9,BLG,0.4667,2.33,2.67


✅ Exported 16 rows to B6:C and G6:H — identical to the preview.


# Champ Analysis

In [ ]:
import pandas as pd, gspread
from gspread_dataframe import set_with_dataframe
from gspread.utils import rowcol_to_a1

# ---------- CONFIG ----------
SERVICE_ACCOUNT_FILE = "worlds-475809-69c16d86976c.json"
SHEET_NAME = "python access"
GRID_RANGE, EXPORT_ROW, EXPORT_COL = "B19:Q35", 8, 3  # C8

roles = {
    "Top":     ["Sion","K'Sante","Ambessa","Rumble","Rek'Sai","Renekton"],
    "Jungle":  ["Vi","Jarvan IV","Xin Zhao","Wukong","Qiyana","Trundle"],
    "Mid":     ["Ryze","Taliyah","Orianna","Aurora","Azir"],
    "ADC":     ["Caitlyn","Yunara","Kai'Sa","Xayah","Ezreal","Corki"],
    "Support": ["Neeko","Alistar","Rakan","Bard","Rell","Braum","Nautilus"],
}

team_map = {
    "Hanwha Life Esports":"HLE","KT Rolster":"KT","Gen.G":"GEN","T1":"T1","G2 Esports":"G2",
    "Fnatic":"FNC","Bilibili Gaming":"BLG","Top Esports":"TES","FlyQuest":"FLY","Movistar KOI":"KOI",
    "CTBC Flying Oyster":"CFO","Anyone's Legend":"AL","PSG Talon":"PSG","Team Secret Whales":"TSW",
    "Vivo Keyd Stars":"VKS","100 Thieves":"100T"
}

# ---------- CONNECT ----------
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sh = gc.open(SHEET_NAME)

# Load matchup grid once (from Top tab) as computed values
grid_vals = sh.worksheet("Top").get(GRID_RANGE, value_render_option="UNFORMATTED_VALUE")
dfg = pd.DataFrame(grid_vals)
col_teams = dfg.iloc[0, 1:].tolist()
row_teams = dfg.iloc[1:, 0].tolist()
mat = dfg.iloc[1:, 1:].apply(pd.to_numeric, errors="coerce")
mat.index, mat.columns = row_teams, col_teams

# ---------- LOAD DATA ----------
df = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv", low_memory=False)
worlds = df[df["league"].astype("string").str.contains(r"\bWLD", case=False, na=False)].copy()
worlds["team_abbr"] = worlds["teamname"].map(team_map)
worlds = worlds.dropna(subset=["team_abbr"])

# opponent extraction per game (safe)
pair_map = (
    worlds.groupby("gameid")["team_abbr"]
    .unique()
    .apply(lambda x: [t for t in x if pd.notna(t)][:2])
    .to_dict()
)
def get_opp(row):
    teams = pair_map.get(row["gameid"], [])
    if len(teams) == 2:
        return teams[1] if teams[0] == row["team_abbr"] else teams[0]
    return None

worlds["opp_abbr"] = worlds.apply(get_opp, axis=1)
worlds = worlds.dropna(subset=["opp_abbr"])

# ---- helpers ----
def number_2dp(ws, start_row, start_col, nrows, col_offset):
    top = rowcol_to_a1(start_row+1, start_col+col_offset)
    bot = rowcol_to_a1(start_row+nrows, start_col+col_offset)
    ws.format(f"{top}:{bot}", {"numberFormat": {"type":"NUMBER","pattern":"0.00"}})

def percent_1dp_RANGE_ONLY_FOR_EWR(ws, sheet, start_row, start_col, nrows, ewr_col_offset):
    """
    Only formats the E(WR) column as percent (1 decimal) on data rows.
    """
    sheet_id = ws.id
    # zero-based indices for Sheets API
    header_row_0 = start_row - 1
    data_r0 = header_row_0 + 1    # first data row (exclude header)
    data_r1 = data_r0 + nrows     # FIX: include last row
    table_c0 = start_col - 1  # convert to 0-based

    col_idx = ewr_col_offset - 1

    def _grid(sheet_id, r0, c0, r1, c1):
        return {"sheetId": sheet_id,
                "startRowIndex": r0, "endRowIndex": r1,
                "startColumnIndex": c0, "endColumnIndex": c1}

    ewr_range = _grid(sheet_id, data_r0, table_c0 + col_idx, data_r1, table_c0 + col_idx + 1)

    sheet.batch_update({
        "requests": [{
            "repeatCell": {
                "range": ewr_range,
                "cell": {"userEnteredFormat": {"numberFormat": {"type":"PERCENT","pattern":"0.0%"}}},
                "fields": "userEnteredFormat.numberFormat"
            }
        }]
    })

def add_wve_row_colors(ws, sheet_id, start_row, start_col, nrows, ncols, wve_col_offset):
    if nrows <= 0: return
    data_start = start_row + 1
    grid_range = {
        "sheetId": sheet_id,
        "startRowIndex": data_start-1,
        "endRowIndex":   data_start-1 + nrows,
        "startColumnIndex": start_col-1,
        "endColumnIndex":   start_col-1 + ncols
    }
    wve_col_idx = start_col + wve_col_offset - 1
    wve_col_letter = "".join(filter(str.isalpha, rowcol_to_a1(1, wve_col_idx)))
    pos_formula, neg_formula = f"=${wve_col_letter}{data_start}>0", f"=${wve_col_letter}{data_start}<0"
    zero_formula = f"=${wve_col_letter}{data_start}=0"
    green = {"red":0.60,"green":0.90,"blue":0.60}
    red   = {"red":0.95,"green":0.55,"blue":0.55}
    yellow= {"red":1.00,"green":0.90,"blue":0.40}
    sh.batch_update({"requests":[
        {"addConditionalFormatRule":{"rule":{"ranges":[grid_range],
         "booleanRule":{"condition":{"type":"CUSTOM_FORMULA","values":[{"userEnteredValue":pos_formula}]},
         "format":{"backgroundColor":green}}},"index":0}},
        {"addConditionalFormatRule":{"rule":{"ranges":[grid_range],
         "booleanRule":{"condition":{"type":"CUSTOM_FORMULA","values":[{"userEnteredValue":neg_formula}]},
         "format":{"backgroundColor":red}}},"index":0}},
        {"addConditionalFormatRule":{"rule":{"ranges":[grid_range],
         "booleanRule":{"condition":{"type":"CUSTOM_FORMULA","values":[{"userEnteredValue":zero_formula}]},
         "format":{"backgroundColor":yellow}}},"index":0}}
    ]})

# ---------- ROLE PROCESSOR ----------
def process_role(role_name, champs):
    ws = sh.worksheet(role_name)
    df_r = worlds[worlds["champion"].isin(champs)].copy()

    # expected prob from grid (row=opponent, col=team)
    def exp_prob(r):
        oa, ta = r["opp_abbr"], r["team_abbr"]
        if oa in mat.index and ta in mat.columns:
            return mat.loc[oa, ta]
        return pd.NA

    df_r["expected_win_prob"] = pd.to_numeric(df_r.apply(exp_prob, axis=1), errors="coerce")
    df_r["result"] = pd.to_numeric(df_r["result"].astype("string").str.strip(), errors="coerce")
    df_r["actual_win"] = (df_r["result"] == 1).fillna(False).astype(int)

    s = (
        df_r.groupby("champion")
            .agg(Games=("champion","count"),
                 Wins=("actual_win","sum"),
                 EWR=("expected_win_prob","sum"))
            .reset_index()
    )

    # Metrics
    s["E(W)"]    = s["EWR"]                  # unrounded expected wins (display 2dp)
    s["E(W)_r"]  = s["EWR"].round(0)         # rounded for WvE calc
    s["E(WR)"]   = s["EWR"] / s["Games"]     # unrounded expected WR
    s["WR"]      = s["Wins"] / s["Games"]
    s["WRvE"]    = s["WR"] - s["E(WR)"]
    s["Losses"]  = s["Games"] - s["Wins"]
    s["WvE"]     = s["Wins"] - s["E(W)_r"]

    # Sort by WvE desc, tie-breaker WRvE desc
    s = s.rename(columns={"champion":"Champions"})[
        ["Champions","E(W)","E(WR)","WR","WRvE","Games","Wins","Losses","WvE"]
    ].sort_values(by=["WvE","WRvE"], ascending=[False, False])

    # Write table at C8
    set_with_dataframe(ws, s, row=EXPORT_ROW, col=EXPORT_COL,
                       include_index=False, include_column_header=True, resize=False)

    # Formats
    nrows, ncols = s.shape
    number_2dp(ws, EXPORT_ROW, EXPORT_COL, nrows, s.columns.get_loc("E(W)") + 1)   # E(W) → 2dp
    percent_1dp_RANGE_ONLY_FOR_EWR(ws, sh, EXPORT_ROW, EXPORT_COL, nrows, s.columns.get_loc("E(WR)") + 1)

    # Row colors by WvE
    add_wve_row_colors(ws, ws.id, EXPORT_ROW, EXPORT_COL, nrows, ncols,
                       wve_col_offset=s.columns.get_loc("WvE") + 1)

    print(f"✅ {role_name} exported ({len(s)} champs) — E(WR) forced to % format")


# ---------- RUN ALL ROLES ----------
for role_name, champs in roles.items():
    process_role(role_name, champs)


✅ Top exported (6 champs) — E(WR) forced to % format
✅ Jungle exported (6 champs) — E(WR) forced to % format
✅ Mid exported (5 champs) — E(WR) forced to % format
✅ ADC exported (6 champs) — E(WR) forced to % format
✅ Support exported (7 champs) — E(WR) forced to % format
